# Análisis vendedores RECESA

Este notebook transforma los archivos de ventas de **Allan** y **Fabiola** en una base maestra limpia para comparar desempeño por vendedor.

Archivos esperados en la misma carpeta del notebook:

- `Ventas redes RECESA Allan.xlsx`
- `Ventas redes RECESA Fabiola.xlsx`

Salida generada:

- `Base_Maestra_Vendedores_RECESA.xlsx`


In [10]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

## 1. Configuración de archivos

In [11]:
# =========================
# ARCHIVOS Y HOJAS
# =========================

archivo_allan = "Ventas redes RECESA Allan.xlsx"
archivo_fabiola = "Ventas redes RECESA Fabiola.xlsx"

hoja_allan = "Ventas_Recesa_Allan"
hoja_fabiola = "Ventas_Recesa_Fabiola"

archivos = {
    "Allan": {
        "archivo": archivo_allan,
        "hoja": hoja_allan
    },
    "Fabiola": {
        "archivo": archivo_fabiola,
        "hoja": hoja_fabiola
    }
}

for vendedor, info in archivos.items():
    if not Path(info["archivo"]).exists():
        raise FileNotFoundError(f"No encontré el archivo: {info['archivo']}")

print("Archivos encontrados correctamente")

Archivos encontrados correctamente


## 2. Leer archivos originales

In [12]:
df_allan_raw = pd.read_excel(
    archivo_allan,
    sheet_name=hoja_allan,
    header=None
)

df_fabiola_raw = pd.read_excel(
    archivo_fabiola,
    sheet_name=hoja_fabiola,
    header=None
)

print("Allan:", df_allan_raw.shape)
print("Fabiola:", df_fabiola_raw.shape)

df_allan_raw.head(8)

Allan: (889, 29)
Fabiola: (569, 46)


,0,1,2,3,4,5,6,7,8,9,...,19,20,21,22,23,24,25,26,27,28
0,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,PRODUCTO TERMINADO / BALANCIN,PRODUCTO TERMINADO / HOJAS,PRODUCTO TERMINADO / LAÑA,PRODUCTO TERMINADO / PERNO,PRODUCTO TERMINADO / PERNO ESTRUCTURAL,PRODUCTO TERMINADO / RESORTAJE,PRODUCTO TERMINADO / TORNILLO CENTRO,REPUESTOS Y ACCESORIOS / ACCESORIO,REPUESTOS Y ACCESORIOS / AMORTIGUADOR,...,REPUESTOS Y ACCESORIOS / PLANCHA,REPUESTOS Y ACCESORIOS / RESORTES,REPUESTOS Y ACCESORIOS / ROLDANA,REPUESTOS Y ACCESORIOS / ROLDANA / TORNILLERIA,REPUESTOS Y ACCESORIOS / SUSPENSION,REPUESTOS Y ACCESORIOS / TRICKET,REPUESTOS Y ACCESORIOS / TUERCA,REPUESTOS Y ACCESORIOS / TUERCA DE RUEDA,SERVICIOS Y REPARACIONES / SERVICIOS,NaN
2,NaN,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,...,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible,Total imponible
3,Total,542.11,1232640.29,190480.42,37505.32,41182.09,276912.12,35033.53,10016.68,5384.68,...,10108.49,1142.86,3027.1,20433.36,8670.18,3679.6,3630.87,2033.92,27117.73,2141977.07
4,PRODUCTO TERMINADO / BALANCIN,542.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,542.11
5,[RL-2601-A] BALANCIN P/CARRETON ( JU...,542.11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,542.11
6,PRODUCTO TERMINADO / HOJAS,NaN,1232640.29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1232640.29
7,[02011313-A] CARRETON 1a 13 3/8X13 3...,NaN,811.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,811.8


In [13]:
df_fabiola_raw.head(8)

,0,1,2,3,4,5,6,7,8,9,...,36,37,38,39,40,41,42,43,44,45
0,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,mayo 2025,NaN,NaN,junio 2025,NaN,NaN,julio 2025,NaN,NaN,...,NaN,mayo 2026,NaN,NaN,junio 2026,NaN,NaN,NaN,NaN,NaN
2,NaN,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,...,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio,Total,Cantidad de producto,Precio promedio
3,Total,955469.19,181385,4.703242,108852.11,17235,5.639073,591992.67,238130,2.219649,...,2.867847,150092.88,21026,6.373606,77634.48,26505,2.615223,3134713.62,879156,3.183567
4,PRODUCTO TERMINADO / HOJAS,546.61,1,488.04,3238.89,11,262.896364,NaN,NaN,NaN,...,NaN,2111.59,3,628.45,NaN,NaN,NaN,11463.9,91,112.479231
5,[01010325-M] CARRETON 1a 2.1/2X25 1....,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3456,72,42.857083
6,[21012229-B] TOYOTA HILUX 90 4WD TRA...,NaN,NaN,NaN,726.47,3,216.21,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,726.47,3,216.21
7,[25HL1417-A] TOYOTA HILUX AUXILIAR 5...,NaN,NaN,NaN,802.73,3,238.906667,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,802.73,3,238.906667


## 3. Funciones de limpieza y transformación

La estructura de estos Excel es:

- Fila 1: meses
- Fila 2: Total / Cantidad de producto / Precio promedio
- Fila 3: totales generales
- Fila 4 en adelante: categorías y productos

El código toma las filas con formato `[CODIGO] Descripción` como productos reales. Las filas como `PRODUCTO TERMINADO / HOJAS` se usan como categoría, pero no entran como producto.


In [14]:
def extraer_codigo_descripcion(texto):
    texto = str(texto).strip()

    match = re.search(r"\[(.*?)\]", texto)

    if match:
        codigo = match.group(1).strip()
        descripcion = re.sub(r"\[.*?\]", "", texto).strip()
        return pd.Series([codigo, descripcion])

    return pd.Series(["", texto])


def limpiar_categoria(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip()
    texto_upper = texto.upper()

    if texto_upper.startswith("PRODUCTO TERMINADO"):
        if "/" in texto:
            return texto.split("/", 1)[1].strip()
        return texto.replace("PRODUCTO TERMINADO", "").strip()

    return None


def transformar_redes(df_raw, vendedor):
    df = df_raw.copy()

    producto_col = 0
    bloques = []

    # Los bloques de meses van cada 3 columnas: Total, Cantidad, Precio promedio
    for col in range(1, df.shape[1], 3):
        if col + 2 >= df.shape[1]:
            continue

        mes = str(df.iloc[1, col]).strip().lower()

        if mes in ["nan", "none", ""]:
            continue

        bloques.append({
            "mes": mes,
            "total": col,
            "cantidad": col + 1,
            "precio": col + 2
        })

    # Desde la fila 4 empiezan categorías y productos
    df_productos = df.iloc[4:].copy().reset_index(drop=True)

    # Detectar categoría y arrastrarla hacia abajo
    df_productos["Categoria_Row"] = df_productos[producto_col].apply(limpiar_categoria)
    df_productos["Categoria"] = df_productos["Categoria_Row"].ffill().fillna("SIN CATEGORIA")

    # Quedarse solo con filas que tienen código entre corchetes
    mask_producto = df_productos[producto_col].astype(str).str.contains(r"\[.*?\]", regex=True, na=False)
    df_productos = df_productos[mask_producto].copy().reset_index(drop=True)

    cod_desc = df_productos[producto_col].apply(extraer_codigo_descripcion)
    cod_desc.columns = ["Codigo", "Descripcion"]

    registros = []

    for bloque in bloques:
        temp = pd.DataFrame(index=df_productos.index)

        temp["Empresa"] = "RECESA"
        temp["Vendedor"] = vendedor
        temp["Categoria"] = df_productos["Categoria"]
        temp["Codigo"] = cod_desc["Codigo"]
        temp["Descripcion"] = cod_desc["Descripcion"]
        temp["Mes"] = bloque["mes"]

        temp["Total"] = pd.to_numeric(
            df_productos[bloque["total"]],
            errors="coerce"
        ).fillna(0)

        temp["Cantidad"] = pd.to_numeric(
            df_productos[bloque["cantidad"]],
            errors="coerce"
        ).fillna(0)

        temp["Precio_Promedio"] = pd.to_numeric(
            df_productos[bloque["precio"]],
            errors="coerce"
        ).fillna(0)

        registros.append(temp)

    df_final = pd.concat(registros, ignore_index=True)

    # Quitar filas sin movimiento en el mes
    df_final = df_final[
        (df_final["Total"] != 0) |
        (df_final["Cantidad"] != 0)
    ].reset_index(drop=True)

    return df_final

## 4. Transformar Allan y Fabiola

In [15]:
df_allan = transformar_redes(df_allan_raw, "Allan")
df_fabiola = transformar_redes(df_fabiola_raw, "Fabiola")

df_total_vendedores = pd.concat(
    [df_allan, df_fabiola],
    ignore_index=True
)

print("Filas Allan:", len(df_allan))
print("Filas Fabiola:", len(df_fabiola))
print("Filas totales:", len(df_total_vendedores))

df_total_vendedores.head(20)

Filas Allan: 624
Filas Fabiola: 1303
Filas totales: 1927


,Empresa,Vendedor,Categoria,Codigo,Descripcion,Mes,Total,Cantidad,Precio_Promedio
0,RECESA,Allan,BALANCIN,RL-2601-A,BALANCIN P/CARRETON ( JUEGO ),producto terminado / balancin,542.11,0.00,0.0
1,RECESA,Allan,HOJAS,02011313-A,CARRETON 1a 13 3/8X13 3/8 R 3/4,producto terminado / balancin,0.00,811.80,0.0
2,RECESA,Allan,HOJAS,02021313-A,CARRETON TRA 2a 13X13 M/R.,producto terminado / balancin,0.00,953.32,0.0
3,RECESA,Allan,HOJAS,02HL0505-A,H LISA 1 3/4 X 3/8 X 5 X 5,producto terminado / balancin,0.00,249.94,0.0
4,RECESA,Allan,HOJAS,02HL0808-A,H LISA 1 3/4 X 3/8 X 8 X 8,producto terminado / balancin,0.00,402.01,0.0
5,RECESA,Allan,HOJAS,02HL1111-M,H LISA 1 3/4 X 3/8 X 11 X 11,producto terminado / balancin,0.00,552.85,0.0
6,RECESA,Allan,HOJAS,10011919-B,SUZUKI DEL 1a,producto terminado / balancin,0.00,1057.35,0.0
7,RECESA,Allan,HOJAS,10012020-A,SUZUKI SAMURAY TRA 1a 19 75X19 75,producto terminado / balancin,0.00,285.04,0.0
8,RECESA,Allan,HOJAS,10012020-B,SUZUKI TRA 1a,producto terminado / balancin,0.00,475.47,0.0
9,RECESA,Allan,HOJAS,10012020-O,CHEVROLET CMV TRA 1a,producto terminado / balancin,0.00,5213.82,0.0


## 5. Ordenar meses

In [16]:
meses_num = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "setiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

# Normalizar columna Mes
mes_normalizado = (
    df_total_vendedores["Mes"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Extraer número de mes
df_total_vendedores["Numero_Mes"] = (
    mes_normalizado
    .str.extract(r"([a-zA-Záéíóúñ]+)")[0]
    .map(meses_num)
)

# Extraer año, permitiendo valores vacíos sin romper
df_total_vendedores["Año"] = (
    mes_normalizado
    .str.extract(r"(\d{4})")[0]
    .astype("Int64")
)

# Texto limpio del mes
df_total_vendedores["Mes_Texto"] = (
    mes_normalizado
    .str.extract(r"([a-zA-Záéíóúñ]+)")[0]
)

# Crear orden YYYYMM
df_total_vendedores["Orden_Mes"] = (
    df_total_vendedores["Año"] * 100 +
    df_total_vendedores["Numero_Mes"]
)

# Opcional: revisar filas donde no detectó mes o año
df_fechas_invalidas = df_total_vendedores[
    df_total_vendedores["Numero_Mes"].isna() |
    df_total_vendedores["Año"].isna()
]

display(df_fechas_invalidas[["Mes", "Mes_Texto", "Numero_Mes", "Año"]].drop_duplicates())

# Ordenar
df_total_vendedores = df_total_vendedores.sort_values(
    ["Orden_Mes", "Vendedor", "Categoria", "Codigo"],
    na_position="last"
).reset_index(drop=True)

df_total_vendedores.head(20)

,Mes,Mes_Texto,Numero_Mes,Año
0,producto terminado / balancin,producto,NaN,<NA>
337,producto terminado / perno,producto,NaN,<NA>
369,producto terminado / tornillo centro,producto,NaN,<NA>
395,repuestos y accesorios / barra,repuestos,NaN,<NA>
489,repuestos y accesorios / esparrago,repuestos,NaN,<NA>
518,repuestos y accesorios / hule,repuestos,NaN,<NA>
521,repuestos y accesorios / plancha,repuestos,NaN,<NA>
528,repuestos y accesorios / roldana / tornilleria,repuestos,NaN,<NA>
606,repuestos y accesorios / tuerca,repuestos,NaN,<NA>


,Empresa,Vendedor,Categoria,Codigo,Descripcion,Mes,Total,Cantidad,Precio_Promedio,Numero_Mes,Año,Mes_Texto,Orden_Mes
0,RECESA,Fabiola,HOJAS,76011721-A,TANDEM CORTA TRA 1a 4X9/16 R 1 5/8,mayo 2025,546.61,1.0,488.040000,5.0,2025,mayo,202505.0
1,RECESA,Fabiola,LAÑA,LCC34418,LANA CUADRADA 3/4 X 4 X 18 CON TUERCA,mayo 2025,1607.42,10.0,143.520000,5.0,2025,mayo,202505.0
2,RECESA,Fabiola,LAÑA,LCC58422,LANA CUADRADA 5/8 X 4 X 22 CON TUERCA,mayo 2025,1589.73,10.0,141.940000,5.0,2025,mayo,202505.0
3,RECESA,Fabiola,LAÑA,LCC91638,LANA CUADRADA 9/16 X 3 X 8 CON TUERCA,mayo 2025,95.00,1.0,84.820000,5.0,2025,mayo,202505.0
4,RECESA,Fabiola,LAÑA,LRC7851416,LANA REDONDA 7/8 X 5 1/4 X 16 CON TUERCA,mayo 2025,520.00,2.0,232.145000,5.0,2025,mayo,202505.0
5,RECESA,Fabiola,PERNO,PEL-0321,PERNO LISO TANDEM 1 3/8 X 6 7/8 2C/REDON (SP-1...,mayo 2025,163.62,1.0,146.090000,5.0,2025,mayo,202505.0
6,RECESA,Fabiola,PERNO ESTRUCTURAL,PSM,PERNO S/MUESTRA,mayo 2025,347321.09,3657.0,84.798502,5.0,2025,mayo,202505.0
7,RECESA,Fabiola,TORNILLO CENTRO,AC-9407,ACEITE INNOVAOIL SEMI SINTETICO SAE 4T 10W40 (...,mayo 2025,100.00,2.0,44.645000,5.0,2025,mayo,202505.0
8,RECESA,Fabiola,TORNILLO CENTRO,AC-9410,ACEITE INNOVAOIL SEMI SINTETICO SAE 20W50 LITRO,mayo 2025,163.20,5.0,29.144000,5.0,2025,mayo,202505.0
9,RECESA,Fabiola,TORNILLO CENTRO,AC-9412,ACEITE INNOVAOIL SEMI SINTETICO SAE 20W50 (GALON),mayo 2025,542.26,4.0,121.040000,5.0,2025,mayo,202505.0


## 6. Resumen general por vendedor

In [17]:
resumen_vendedor = (
    df_total_vendedores
    .groupby("Vendedor", as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
)

resumen_vendedor["Precio_Promedio"] = (
    resumen_vendedor["Total"] /
    resumen_vendedor["Cantidad"]
).replace([np.inf, -np.inf], 0).fillna(0)

resumen_vendedor

,Vendedor,Total,Cantidad,Precio_Promedio
0,Allan,131100.77,1479094.46,0.088636
1,Fabiola,3134713.62,879156.00,3.565594


## 7. Resumen mensual por vendedor

In [18]:
resumen_mensual = (
    df_total_vendedores
    .groupby(["Orden_Mes", "Mes_Texto", "Vendedor"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values(["Orden_Mes", "Vendedor"])
)

resumen_mensual

,Orden_Mes,Mes_Texto,Vendedor,Total,Cantidad
0,202505.0,mayo,Fabiola,955469.19,181385.0
1,202506.0,junio,Fabiola,108852.11,17235.0
2,202507.0,julio,Fabiola,591992.67,238130.0
3,202508.0,agosto,Fabiola,80704.94,30934.0
4,202509.0,septiembre,Fabiola,229117.92,37852.0
5,202510.0,octubre,Fabiola,94242.35,43693.0
6,202511.0,noviembre,Fabiola,100315.81,30901.0
7,202512.0,diciembre,Fabiola,188727.64,22621.0
8,202601.0,enero,Fabiola,142054.67,36173.0
9,202602.0,febrero,Fabiola,140690.88,127488.0


## 8. Resumen por categoría y vendedor

In [19]:
resumen_categoria = (
    df_total_vendedores
    .groupby(["Vendedor", "Categoria"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values(["Vendedor", "Total"], ascending=[True, False])
)

resumen_categoria

,Vendedor,Categoria,Total,Cantidad
4,Allan,TORNILLO CENTRO,93053.34,205272.08
2,Allan,PERNO,37505.32,0.00
0,Allan,BALANCIN,542.11,0.00
1,Allan,HOJAS,0.00,1232640.29
3,Allan,PERNO ESTRUCTURAL,0.00,41182.09
10,Fabiola,TORNILLO CENTRO,1534658.78,731894.00
8,Fabiola,PERNO ESTRUCTURAL,1240022.46,23252.00
9,Fabiola,TORNILLERIA Y FERRETERIA,316418.93,122911.00
6,Fabiola,LAÑA,23344.60,210.00
5,Fabiola,HOJAS,11463.90,91.00


## 9. Top productos general

In [20]:
top_productos = (
    df_total_vendedores
    .groupby(["Codigo", "Descripcion"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values("Total", ascending=False)
)

top_productos.head(20)

,Codigo,Descripcion,Total,Cantidad
530,PSM,PERNO S/MUESTRA,1239506.58,51817.1
740,ROA32534212,TOR. HEX. A325 3/4 X 2.1/2,425953.20,40115.0
1043,TROA32534,TUERCA HEX. A325 3/4,172240.94,64210.0
423,EL-2423,ELECTRODO 7018 1/8 (X LIBRA),95294.00,5800.0
1058,VA-3416,"ELECTRODO 6013 3/32"" (LIBRA)",73356.80,5731.0
1045,TROA32578,TUERCA HEX. A325 7/8,65078.75,11880.0
1039,TROA3251,"TUERCA HEX. A325 1""",55769.60,7937.0
415,EL-2415,CLAVO ACERADO TIPO Cc27 CON ROLDANA ( SIN FULM...,53200.00,70000.0
769,RPAI34,"ROLDANA DE 3/4"" ACERO INOX.",47850.00,15000.0
777,RPF43634,ROLDANA F436 3/4,38135.60,48684.0


## 10. Top productos por vendedor

In [21]:
top_productos_vendedor = (
    df_total_vendedores
    .groupby(["Vendedor", "Codigo", "Descripcion"], as_index=False)
    .agg({
        "Total": "sum",
        "Cantidad": "sum"
    })
    .sort_values(["Vendedor", "Total"], ascending=[True, False])
)

top_productos_vendedor.head(30)

,Vendedor,Codigo,Descripcion,Total,Cantidad
401,Allan,PZM,PIEZA S/MUESTRA,12580.35,0.0
572,Allan,TC384,TORNILLO CENTRO 3/8 X 4,8265.30,0.0
611,Allan,TROA325114,TUERCA HEX. A325 1.1/4,5620.67,0.0
395,Allan,PL-0005,PLANCHA P/LANA AG. 5/8 (CAMA 3 1/2 ),4965.93,0.0
566,Allan,TC1210,TORNILLO CENTRO 1/2 X 10,4522.61,0.0
419,Allan,RL-2720,ESPARRAGO 04042-1004 (R) TRA HINO KL-KR,4129.45,0.0
587,Allan,TCE128,TORNILLO CENTRO ESP 1/2 X 8 (INTER),3921.47,0.0
354,Allan,BR13,BARRA ROSCADA 1 X 3MTS,3673.96,0.0
465,Allan,RPF436114,"ROLDANA F436 1.1/4""",3652.16,0.0
392,Allan,PET-2573,PERNO C/TUERCA R F 1 X 10,3341.75,0.0


## 11. Validaciones rápidas

In [22]:
print("VENTAS TOTALES:", round(df_total_vendedores["Total"].sum(), 2))
print("UNIDADES TOTALES:", round(df_total_vendedores["Cantidad"].sum(), 2))

print("\nMESES DETECTADOS:")
print(
    df_total_vendedores[["Mes_Texto", "Orden_Mes"]]
    .drop_duplicates()
    .sort_values("Orden_Mes")
    .to_string(index=False)
)

print("\nVENDEDORES DETECTADOS:")
print(df_total_vendedores["Vendedor"].unique())

VENTAS TOTALES: 3265814.39
UNIDADES TOTALES: 2358250.46

MESES DETECTADOS:
 Mes_Texto  Orden_Mes
      mayo   202505.0
     junio   202506.0
     julio   202507.0
    agosto   202508.0
septiembre   202509.0
   octubre   202510.0
 noviembre   202511.0
 diciembre   202512.0
     enero   202601.0
   febrero   202602.0
     marzo   202603.0
     abril   202604.0
      mayo   202605.0
     junio   202606.0
  producto       <NA>
 repuestos       <NA>

VENDEDORES DETECTADOS:
['Fabiola' 'Allan']


## 12. Exportar Excel final

In [23]:
with pd.ExcelWriter(
    "Base_Maestra_Vendedores_RECESA.xlsx",
    engine="openpyxl"
) as writer:

    df_total_vendedores.to_excel(
        writer,
        sheet_name="Base Maestra",
        index=False
    )

    resumen_vendedor.to_excel(
        writer,
        sheet_name="Resumen Vendedor",
        index=False
    )

    resumen_mensual.to_excel(
        writer,
        sheet_name="Resumen Mensual",
        index=False
    )

    resumen_categoria.to_excel(
        writer,
        sheet_name="Resumen Categoria",
        index=False
    )

    top_productos.to_excel(
        writer,
        sheet_name="Top Productos",
        index=False
    )

    top_productos_vendedor.to_excel(
        writer,
        sheet_name="Top por Vendedor",
        index=False
    )

print("Archivo generado correctamente: Base_Maestra_Vendedores_RECESA.xlsx")

Archivo generado correctamente: Base_Maestra_Vendedores_RECESA.xlsx
